# Export SymTRELLIS Mapper Checkpoint

Temporary notebook for exporting a trainer checkpoint into an inference-only `config.json` and `model.safetensors`.

The trainer checkpoint stores the training config, not the mapper dataclass config. This notebook reconstructs the mapper config from `model_backend`, `model_scale`, `latent_dim`, `lowrank_rank`, and `attention_backend`, then verifies that the exported safetensors can be loaded with `strict=True`.

## Output Schema

The JSON is a SymTRELLIS-specific inference config stored in `config.json`. It contains only the information needed to instantiate the mapper and load `model.safetensors`.

Important fields:

- `format` / `format_version`: loader-facing schema identity.
- `model_backend`: `swin3d` or `neighbor_graph`.
- `model_class` / `model_config_class`: class names needed by a custom loader.
- `model_config`: full reconstructed dataclass config used to instantiate the mapper.
- `weights`: safetensors filename and parameter summary.

In [25]:
from pathlib import Path

# Fill these before running.
CHECKPOINT_PATH = Path("/mnt/nvmefs/Projects/symtrellis_train/neighbor_graph_ss_mapper_train/checkpoints/epoch_0099.pt")
OUTPUT_DIR = Path("/mnt/nvmefs/Projects/symtrellis_hf_repo/trellis2/sparse_structure/neighbor_graph/pretrain")
MODEL_NAME = "trellis2_shape_mapper_neighbor_graph_pretrain"

# Use None to preserve the training attention backend. For release/inference,
# flash_attn is usually the intended Swin3D backend when using fp16/bf16.
ATTENTION_BACKEND_FOR_EXPORT = None

assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
import json
from dataclasses import asdict, replace
from typing import Any

import torch
from safetensors.torch import load_file, save_file

from symtrellis.mapper import (
    NeighborGraphLatentMapper,
    Swin3DLatentMapper,
    neighbor_graph_latent_mapper_config,
    swin_3d_latent_mapper_config,
)


def to_jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [to_jsonable(v) for v in value]
    if isinstance(value, list):
        return [to_jsonable(v) for v in value]
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    return value


checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
training_config = dict(checkpoint["config"])
state_dict = checkpoint["model"]

model_backend = training_config["model_backend"]
model_scale = training_config["model_scale"]
latent_dim = int(training_config["latent_dim"])
lowrank_rank = int(training_config["lowrank_rank"])

if model_backend == "swin3d":
    model_config = swin_3d_latent_mapper_config(
        scale=model_scale,
        latent_dim=latent_dim,
        lowrank_rank=lowrank_rank,
    )
    attention_backend = ATTENTION_BACKEND_FOR_EXPORT or training_config.get("attention_backend", model_config.attn_backend)
    model_config = replace(model_config, attn_backend=attention_backend)
    model_class = Swin3DLatentMapper
elif model_backend == "neighbor_graph":
    model_config = neighbor_graph_latent_mapper_config(
        scale=model_scale,
        latent_dim=latent_dim,
        lowrank_rank=lowrank_rank,
    )
    model_class = NeighborGraphLatentMapper
else:
    raise ValueError(f"Unknown model_backend: {model_backend}")

model_config_dict = to_jsonable(asdict(model_config))
model = model_class(model_config)
missing, unexpected = model.load_state_dict(state_dict, strict=True)
assert not missing and not unexpected

parameter_count = sum(t.numel() for t in state_dict.values())
state_dtypes = sorted({str(t.dtype).replace("torch.", "") for t in state_dict.values()})

print("model_backend:", model_backend)
print("model_class:", model_class.__name__)
print("model_scale:", model_scale)
print("latent_dim:", latent_dim)
print("lowrank_rank:", lowrank_rank)
print("parameter_count:", parameter_count)
print("state_dtypes:", state_dtypes)

model_backend: neighbor_graph
model_class: NeighborGraphLatentMapper
model_scale: base
latent_dim: 8
lowrank_rank: 64
parameter_count: 13053396
state_dtypes: ['float32']


In [27]:
artifact_config = {
    "format": "symtrellis.mapper",
    "format_version": 1,
    "name": MODEL_NAME,
    "model_backend": model_backend,
    "model_scale": model_scale,
    "model_class": model_class.__name__,
    "model_config_class": type(model_config).__name__,
    "model_config": model_config_dict,
    "weights": {
        "filename": "model.safetensors",
        "format": "safetensors",
        "parameter_count": int(parameter_count),
        "dtypes": state_dtypes,
    },
}

json_path = OUTPUT_DIR / "config.json"
safetensors_path = OUTPUT_DIR / "model.safetensors"

json_path.write_text(json.dumps(artifact_config, indent=2, sort_keys=True) + "\n")
save_file(
    {k: v.detach().cpu().contiguous() for k, v in state_dict.items()},
    safetensors_path,
    metadata={
        "format": "pt",
        "symtrellis_format": "symtrellis.mapper",
        "model_name": MODEL_NAME,
    },
)

print(json_path)
print(safetensors_path)

/mnt/nvmefs/Projects/symtrellis_hf_repo/trellis2/sparse_structure/neighbor_graph/pretrain/config.json
/mnt/nvmefs/Projects/symtrellis_hf_repo/trellis2/sparse_structure/neighbor_graph/pretrain/model.safetensors


In [28]:
# Verify the exported files can reconstruct and strictly load the mapper.
loaded_config = json.loads(json_path.read_text())

if loaded_config["model_class"] == "Swin3DLatentMapper":
    from symtrellis.mapper.config import Swin3DLatentMapperConfig

    verify_model = Swin3DLatentMapper(Swin3DLatentMapperConfig(**loaded_config["model_config"]))
elif loaded_config["model_class"] == "NeighborGraphLatentMapper":
    from symtrellis.mapper.config import NeighborGraphLatentMapperConfig

    verify_model = NeighborGraphLatentMapper(NeighborGraphLatentMapperConfig(**loaded_config["model_config"]))
else:
    raise ValueError(loaded_config["model_class"])

loaded_state = load_file(safetensors_path)
missing, unexpected = verify_model.load_state_dict(loaded_state, strict=True)
assert not missing and not unexpected

print("export verified")

export verified
